# Sharp RDD: Education Policy at a Test Score Cutoff

**Module 05 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement a sharp RDD from raw data
2. Visualise the discontinuity at the cutoff using bin scatter plots
3. Estimate the treatment effect using local linear regression
4. Run the density test for manipulation
5. Check covariate balance at the cutoff

## Context

We study the effect of a merit scholarship on university GPA. Students who score at least 700 on an entrance exam receive a scholarship; those below do not. The design is a canonical sharp RDD:

- **Running variable:** entrance exam score
- **Cutoff:** 700 points
- **Treatment:** merit scholarship (yes/no)
- **Outcome:** first-year university GPA

The key question: does receiving the scholarship cause students to achieve higher GPA, or do students who score just above 700 achieve higher GPA simply because they are slightly stronger academically?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(700)  # the cutoff
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Generate Scholarship RDD Data

We simulate a dataset with the following structure:
- Exam score: roughly normally distributed around 700
- Scholarship: assigned to students scoring ≥ 700
- True treatment effect: 0.25 GPA points
- GPA function: nonlinear in exam score (quadratic), with a discontinuity at 700

In [ ]:
N = 1500
CUTOFF = 700
TRUE_EFFECT = 0.25  # scholarship raises GPA by 0.25 points

# Exam score: truncated normal around cutoff
score = np.random.normal(700, 60, N)
score = np.clip(score, 450, 950)  # realistic range

# Centre around cutoff (for analysis)
x = score - CUTOFF

# Treatment: sharp cutoff
treated = (score >= CUTOFF).astype(int)

# GPA: quadratic in exam score + treatment effect + noise
# The true regression function is curved (quadratic)
gpa = (2.5
       + 0.008 * x               # linear slope
       - 0.000015 * x**2         # quadratic curvature
       + TRUE_EFFECT * treated   # treatment effect
       + np.random.normal(0, 0.3, N))  # noise
gpa = np.clip(gpa, 0, 4.0)  # GPA range 0-4.0

# Baseline covariates (should not jump at cutoff)
age = np.random.randint(17, 23, N)  # mostly 18-22
female = np.random.binomial(1, 0.52, N)
ses = np.random.normal(50, 15, N)   # socioeconomic status index

df = pd.DataFrame({
    'score': score,
    'x': x,              # centred around cutoff
    'treated': treated,
    'gpa': gpa,
    'age': age,
    'female': female,
    'ses': ses
})

print(f"Dataset: N = {N}, cutoff = {CUTOFF}, true effect = {TRUE_EFFECT}")
print(f"\nTreated: {treated.sum()} ({100*treated.mean():.0f}%)")
print(f"Control: {(1-treated).sum()} ({100*(1-treated.mean()):.0f}%)")
print(f"\nMean GPA: treated = {gpa[treated==1].mean():.3f}, control = {gpa[treated==0].mean():.3f}")
print(f"Raw difference: {gpa[treated==1].mean() - gpa[treated==0].mean():.3f} (confounded by score)")

## 2. Visualise the Discontinuity

The bin scatter plot is the standard visualisation for RDD. We divide the running variable into equally spaced bins and plot the average outcome in each bin.

In [ ]:
def bin_scatter(df, x_col, y_col, cutoff=0, n_bins=40, ax=None):
    """Create a bin scatter plot for RDD visualisation."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 5))
    
    # Create bins
    df = df.copy()
    df['bin'] = pd.cut(df[x_col], bins=n_bins)
    
    # Compute bin means
    bin_means = df.groupby('bin').agg(
        x_mid=(x_col, 'mean'),
        y_mean=(y_col, 'mean'),
        n=(y_col, 'count')
    ).dropna()
    
    # Separate sides
    left = bin_means[bin_means['x_mid'] < cutoff]
    right = bin_means[bin_means['x_mid'] >= cutoff]
    
    ax.scatter(left['x_mid'], left['y_mean'], color='steelblue', s=40, zorder=5, label='Control')
    ax.scatter(right['x_mid'], right['y_mean'], color='darkorange', s=40, zorder=5, label='Treated')
    
    # Fit lines (local linear)
    for side_df, color, h in [(left, 'steelblue', 50), (right, 'darkorange', 50)]:
        x_vals = side_df['x_mid'].values
        y_vals = side_df['y_mean'].values
        if len(x_vals) > 2:
            z = np.polyfit(x_vals, y_vals, 1)
            p = np.poly1d(z)
            x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
            ax.plot(x_line, p(x_line), '-', color=color, linewidth=2, alpha=0.8)
    
    ax.axvline(cutoff, color='red', linewidth=2, linestyle='--', alpha=0.7, label='Cutoff')
    ax.set_xlabel(f'{x_col} (centred at cutoff)')
    ax.set_ylabel(y_col)
    ax.legend()
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full range
bin_scatter(df, 'x', 'gpa', cutoff=0, n_bins=40, ax=axes[0])
axes[0].set_title('RDD: GPA vs Exam Score\n(Full range — bin scatter)')
axes[0].set_xlabel('Exam Score (centred at 700)')
axes[0].set_ylabel('First-Year GPA')

# Right: zoom around cutoff
bandwidth = 150
local_df = df[np.abs(df['x']) <= bandwidth]
bin_scatter(local_df, 'x', 'gpa', cutoff=0, n_bins=20, ax=axes[1])
axes[1].set_title(f'RDD: Zoomed to ±{bandwidth} of Cutoff')
axes[1].set_xlabel('Exam Score (centred at 700)')
axes[1].set_ylabel('First-Year GPA')

plt.tight_layout()
plt.show()

print(f"\nVisual discontinuity apparent at score = {CUTOFF}")

## 3. Local Linear RDD Estimation

We implement local linear regression manually and via OLS to estimate the jump at the cutoff. The treatment effect is the coefficient on the `treated` indicator after controlling for the running variable on both sides.

In [ ]:
def local_linear_rdd(df, outcome, x_col, cutoff, bandwidth):
    """Estimate sharp RDD using local linear regression."""
    # Restrict to bandwidth
    mask = np.abs(df[x_col]) <= bandwidth
    local_df = df[mask].copy()
    local_df['treated'] = (local_df[x_col] >= cutoff).astype(int)
    
    # Local linear regression: different slopes on each side
    formula = f'{outcome} ~ treated + {x_col} + treated:{x_col}'
    model = smf.ols(formula, data=local_df).fit(cov_type='HC1')
    
    tau = model.params['treated']
    se = model.bse['treated']
    ci = model.conf_int().loc['treated'].values
    n_used = mask.sum()
    
    return {
        'estimate': tau, 'se': se,
        'ci_lo': ci[0], 'ci_hi': ci[1],
        'n_obs': n_used, 'model': model
    }

# Estimate across different bandwidths
bandwidths = [50, 80, 100, 120, 150, 200]
print(f"Local Linear RDD Results (True effect = {TRUE_EFFECT})")
print("=" * 70)
print(f"{'Bandwidth':>10} {'Estimate':>10} {'SE':>8} {'95% CI':>18} {'N obs':>7}")
print("-" * 70)

results = []
for h in bandwidths:
    r = local_linear_rdd(df, 'gpa', 'x', 0, h)
    results.append({'bandwidth': h, **r})
    ci_str = f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]"
    print(f"{h:>10} {r['estimate']:>+10.3f} {r['se']:>8.3f} {ci_str:>18} {r['n_obs']:>7}")

print("-" * 70)
print(f"{'True effect':>10} {TRUE_EFFECT:>+10.3f}")

## 4. Density Test (McCrary Test)

Test whether the running variable's density is continuous at the cutoff. A significant discontinuity suggests manipulation.

In [ ]:
# Visual density check: histogram around cutoff
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of running variable
ax = axes[0]
n_below = (df['x'] < 0).sum()
n_above = (df['x'] >= 0).sum()

ax.hist(df[df['x'] < 0]['x'], bins=30, color='steelblue', alpha=0.7,
        label=f'Below cutoff (n={n_below})', density=True)
ax.hist(df[df['x'] >= 0]['x'], bins=30, color='darkorange', alpha=0.7,
        label=f'Above cutoff (n={n_above})', density=True)
ax.axvline(0, color='red', linewidth=2, linestyle='--', label='Cutoff')
ax.set_xlabel('Exam Score (centred at 700)')
ax.set_ylabel('Density')
ax.set_title('Running Variable Distribution\n(Check for bunching at cutoff)')
ax.legend()

# Right: fine bins near cutoff
ax2 = axes[1]
local_x = df[(df['x'] > -100) & (df['x'] < 100)]['x']
ax2.hist(local_x, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
ax2.axvline(0, color='red', linewidth=2, linestyle='--', label='Cutoff')
ax2.set_xlabel('Exam Score (centred at 700, ±100 range)')
ax2.set_ylabel('Count')
ax2.set_title('Density near Cutoff\n(Spike would indicate manipulation)')
ax2.legend()

plt.tight_layout()
plt.show()

# Formal test: compare density just below vs just above cutoff
# Use a local linear approximation of the density
bin_size = 10
bins = pd.cut(df['x'], bins=range(-200, 210, bin_size))
counts = df.groupby(bins).size()

# Left of cutoff bins
left_counts = counts[counts.index.mid < 0].tail(5)
right_counts = counts[counts.index.mid >= 0].head(5)

t_stat, p_val = stats.ttest_ind(left_counts.values, right_counts.values)
print(f"Density continuity test at cutoff:")
print(f"  Mean count (below cutoff, 5 bins): {left_counts.mean():.1f}")
print(f"  Mean count (above cutoff, 5 bins): {right_counts.mean():.1f}")
print(f"  t-statistic: {t_stat:.3f}, p-value: {p_val:.3f}")
if p_val > 0.05:
    print("  No significant discontinuity in density → manipulation unlikely")
else:
    print("  Significant density discontinuity → investigate manipulation")

## 5. Covariate Balance at the Cutoff

Baseline covariates should not jump at the cutoff. We test this by running the same RDD on each covariate as the 'outcome'.

In [ ]:
# Test for covariate balance using RDD at the same cutoff
covariates = ['age', 'female', 'ses']
h_default = 100  # bandwidth for balance test

print(f"Covariate Balance at Cutoff (bandwidth = {h_default})")
print("=" * 60)
print(f"{'Covariate':<12} {'Jump (τ)':>10} {'SE':>8} {'p-value':>10} {'Balanced?':>10}")
print("-" * 60)

for cov in covariates:
    r = local_linear_rdd(df, cov, 'x', 0, h_default)
    # Approximate p-value from t-statistic
    t_stat = r['estimate'] / r['se']
    df_resid = r['n_obs'] - 4  # 4 parameters in local linear with interaction
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_resid))
    status = "OK" if p_val > 0.05 else "PROBLEM"
    print(f"{cov:<12} {r['estimate']:>+10.4f} {r['se']:>8.4f} {p_val:>10.4f} {status:>10}")

print("-" * 60)
print("p > 0.05 for all covariates → baseline characteristics balanced at cutoff")
print("(This supports the continuity assumption)")

## 6. Final RDD Plot

Produce a publication-quality RDD plot showing the discontinuity with a confidence band.

In [ ]:
# Use optimal-ish bandwidth for main result
h_main = 100
main_result = local_linear_rdd(df, 'gpa', 'x', 0, h_main)

fig, ax = plt.subplots(figsize=(12, 6))

# Bin scatter
local_df = df[np.abs(df['x']) <= h_main + 50]
bin_scatter(local_df, 'x', 'gpa', cutoff=0, n_bins=25, ax=ax)

# Annotate the jump
# Get predicted values at cutoff from each side
model = main_result['model']
# At x=0: left side prediction = Intercept, right side = Intercept + treated
y_left = model.params['Intercept']
y_right = model.params['Intercept'] + model.params['treated']

ax.annotate('', xy=(2, y_right), xytext=(2, y_left),
            arrowprops=dict(arrowstyle='<->', color='darkred', lw=2.5))
ax.text(5, (y_left + y_right)/2, f' DiD = {main_result["estimate"]:+.3f}\n GPA pts',
        color='darkred', fontsize=11, va='center')

ax.set_xlabel('Entrance Exam Score (centred at cutoff = 700)', fontsize=12)
ax.set_ylabel('First-Year University GPA', fontsize=12)
ax.set_title(
    f'Sharp RDD: Effect of Merit Scholarship on GPA\n'
    f'Bandwidth = ±{h_main} pts | Estimated effect = {main_result["estimate"]:+.3f} '
    f'[95% CI: {main_result["ci_lo"]:+.3f}, {main_result["ci_hi"]:+.3f}] | '
    f'True effect = {TRUE_EFFECT:+.3f}',
    fontsize=11
)

plt.tight_layout()
plt.savefig('../resources/sharp_rdd.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nFinal estimate: τ = {main_result['estimate']:.3f} (SE = {main_result['se']:.3f})")
print(f"True effect:    τ = {TRUE_EFFECT:.3f}")
print(f"Bias:           {main_result['estimate'] - TRUE_EFFECT:+.3f}")

## 7. Interpretation

The RDD estimate of approximately 0.25 GPA points represents the **Local Average Treatment Effect at the cutoff**: the causal effect of receiving the merit scholarship for students who scored right around 700 on the entrance exam.

**Key points:**
1. This is NOT the effect for all scholarship recipients — it's the effect for students near the cutoff
2. Students far above 700 might benefit differently (possibly less, if the scholarship is redundant for highly motivated students)
3. The raw GPA difference between treated and control is much larger (confounded by exam score), showing why the raw comparison is misleading
4. The estimate is credible because: (a) density test passes, (b) covariates balance at cutoff, (c) the effect is stable across bandwidths

## Self-Check

1. Increase the noise level (`np.random.normal(0, 0.3, N)` → `np.random.normal(0, 0.8, N)`). How does this affect the precision of the RDD estimate at different bandwidths?

2. Add manipulation: make 5% of students just below the cutoff (x ∈ [-20, 0]) have their score inflated to just above (x ∈ [0, 20]). Does the density test detect this? What happens to the RDD estimate?

3. Make the running variable effect more nonlinear by changing the quadratic coefficient from `-0.000015` to `-0.00008`. Does the local linear estimator still recover the true effect? Why or why not?

4. Run the covariate balance test on `ses` after adding a systematic difference: `ses += 3 * treated`. Does the balance test detect this? What would it mean if `ses` jumped at the cutoff?

---
**Next:** `02_rdd_sensitivity.ipynb` — Bandwidth selection and sensitivity analysis